# Fine-tune Gemma 2B with QLoRA
> Parameter-efficient fine-tuning on domain-specific data  
> Model: `google/gemma-2b` · Method: QLoRA (4-bit) + LoRA adapters · Runtime: Colab Free T4

| Step | Module |
|------|--------|
| 1 | Install & import |
| 2 | Config |
| 3 | Dataset |
| 4 | Model (QLoRA + LoRA) |
| 5 | Train |
| 6 | Metrics & plots |
| 7 | Inference demo |

In [ ]:
# ── Check GPU ────────────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── 1. Install ───────────────────────────────────────────────────────────────
!pip install -q \
    torch transformers datasets peft trl \
    bitsandbytes accelerate sentencepiece \
    evaluate rouge-score matplotlib

In [ ]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import os, json, math, warnings
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, DataCollatorForLanguageModeling,
    TrainerCallback, TrainerState, TrainerControl, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer
from datasets import load_dataset
from evaluate import load as load_metric

warnings.filterwarnings('ignore')
print(f'PyTorch  : {torch.__version__}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — switch runtime to T4!"}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 3. Config — edit these two lines when you pick your dataset ──────────────
MODEL_NAME   = 'google/gemma-2b'
DATASET_NAME = 'REPLACE_ME'       # ← e.g. 'tatsu-lab/alpaca'
OUTPUT_DIR   = './outputs'
MAX_SEQ_LEN  = 512
VAL_RATIO    = 0.1
SEED         = 42

# LoRA
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
TARGET_MODS  = ['q_proj', 'k_proj', 'v_proj', 'o_proj']

# Training
EPOCHS       = 3
BATCH_SIZE   = 2
GRAD_ACCUM   = 8     # effective batch = 16
LR           = 2e-4

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config ready.')

In [ ]:
# ── 4. Dataset ───────────────────────────────────────────────────────────────
PROMPT_TEMPLATE = (
    '### Instruction:\n{instruction}\n\n'
    '### Input:\n{input}\n\n'
    '### Response:\n{output}'
)

def format_prompt(ex):
    return {'text': PROMPT_TEMPLATE.format(
        instruction=ex.get('instruction', ''),
        input=ex.get('input', ''),
        output=ex.get('output', ''),
    )}

raw   = load_dataset(DATASET_NAME, split='train')
split = raw.train_test_split(test_size=VAL_RATIO, seed=SEED)
split = split.map(format_prompt, remove_columns=split['train'].column_names)

print(f'Train : {len(split["train"]):,}')
print(f'Val   : {len(split["test"]):,}')
print(f'\nSample:\n{split["train"][0]["text"][:300]}...')

In [ ]:
# ── 5. Tokenizer ─────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, add_eos_token=True
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

def tokenize(batch):
    tokens = tokenizer(
        batch['text'], truncation=True,
        max_length=MAX_SEQ_LEN, padding=False
    )
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

tokenized = split.map(tokenize, batched=True, remove_columns=['text'])
tokenized.set_format('torch')
print('Tokenization done.')

In [ ]:
# ── 6. Load Gemma 2B in 4-bit (QLoRA) ────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False

# Freeze quantized weights — only LoRA adapters will be updated
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Attach LoRA adapters
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODS,
    lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM',
))

model.print_trainable_parameters()

In [ ]:
# ── 7. Loss recorder callback ─────────────────────────────────────────────────
train_losses, eval_losses = [], []

class LossRecorder(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs: return
        step = state.global_step
        if 'loss'      in logs: train_losses.append({'step': step, 'loss': logs['loss']})
        if 'eval_loss' in logs: eval_losses.append({'step': step,  'loss': logs['eval_loss']})

In [ ]:
# ── 8. Train ──────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    dataset_text_field=None,
    max_seq_length=MAX_SEQ_LEN,
    callbacks=[LossRecorder()],
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.001,
        fp16=True,
        logging_steps=10,
        evaluation_strategy='steps',
        eval_steps=50,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        report_to='none',
        seed=SEED,
        group_by_length=True,
        dataloader_pin_memory=False,
    ),
)

trainer.train()

In [ ]:
# ── 9. Save adapter ───────────────────────────────────────────────────────────
ADAPTER_PATH = os.path.join(OUTPUT_DIR, 'lora_adapter')
trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'Adapter saved → {ADAPTER_PATH}')

In [ ]:
# ── 10. Loss curves ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#F8F9FA')
ax.set_facecolor('#F8F9FA')

if train_losses:
    ax.plot([p['step'] for p in train_losses],
            [p['loss'] for p in train_losses],
            color='#5C6BC0', lw=1.8, label='Train loss')
if eval_losses:
    ax.plot([p['step'] for p in eval_losses],
            [p['loss'] for p in eval_losses],
            color='#EF5350', lw=1.8, ls='--', label='Eval loss')

ax.set_xlabel('Step', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Training & Evaluation Loss', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.3f'))
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11. Perplexity ────────────────────────────────────────────────────────────
eval_out   = trainer.evaluate()
eval_loss  = eval_out['eval_loss']
perplexity = math.exp(eval_loss)
print(f'Eval loss  : {eval_loss:.4f}')
print(f'Perplexity : {perplexity:.4f}')

In [ ]:
# ── 12. ROUGE ─────────────────────────────────────────────────────────────────
rouge    = load_metric('rouge')
N_EVAL   = 100
preds, refs = [], []

model.eval()
for ex in list(tokenized['test'])[:N_EVAL]:
    text   = tokenizer.decode(ex['input_ids'], skip_special_tokens=True)
    marker = '### Response:'
    if marker in text:
        prompt = text[:text.index(marker) + len(marker)]
        ref    = text[text.index(marker) + len(marker):].strip()
    else:
        prompt, ref = text, ''

    inp = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inp, max_new_tokens=128, temperature=0.7,
            top_p=0.9, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    pred = tokenizer.decode(out[0][inp['input_ids'].shape[-1]:], skip_special_tokens=True)
    preds.append(pred.strip())
    refs.append(ref)

rouge_scores = rouge.compute(predictions=preds, references=refs)
print('ROUGE scores:')
for k, v in rouge_scores.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# ── 13. ROUGE bar chart ───────────────────────────────────────────────────────
keys   = [k for k in rouge_scores if k.startswith('rouge')]
vals   = [rouge_scores[k] for k in keys]
colors = ['#5C6BC0', '#26A69A', '#EF5350', '#FFA726']

fig, ax = plt.subplots(figsize=(6, 4))
fig.patch.set_facecolor('#F8F9FA')
ax.set_facecolor('#F8F9FA')
bars = ax.bar(keys, vals, color=colors[:len(keys)], width=0.5, edgecolor='white')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{v:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, min(1.0, max(vals) * 1.3))
ax.set_ylabel('Score', fontsize=11)
ax.set_title('ROUGE Scores — validation set', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/rouge_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 14. Save eval report ──────────────────────────────────────────────────────
report = {
    'perplexity': round(perplexity, 4),
    **{k: round(v, 4) for k, v in rouge_scores.items()}
}
with open(f'{OUTPUT_DIR}/eval_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

In [ ]:
# ── 15. Inference demo ────────────────────────────────────────────────────────
def generate(instruction, input_text=''):
    prompt = (
        f'### Instruction:\n{instruction}\n\n'
        f'### Input:\n{input_text}\n\n'
        f'### Response:\n'
    )
    inp = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inp['input_ids'].shape[-1]:],
        skip_special_tokens=True
    ).strip()

# ✏️  Replace with a prompt relevant to your domain
response = generate(
    instruction='Explain the concept of gradient descent in simple terms.',
    input_text='',
)
print('Response:\n', response)

---
## Results summary

| Metric | Value |
|--------|-------|
| Model | Gemma 2B |
| Method | QLoRA 4-bit + LoRA (r=16) |
| Trainable params | ~1% of total |
| Perplexity | see `eval_report.json` |
| ROUGE-1 | see `eval_report.json` |
| ROUGE-L | see `eval_report.json` |

**Outputs saved in `./outputs/`**  
`lora_adapter/` · `loss_curves.png` · `rouge_scores.png` · `eval_report.json` · `metrics_history.json`